In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully.')

Mounted at /content/drive
Drive mounted successfully.


In [ ]:
# Imports

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import numpy as np

# Use a non-interactive backend — safe for Colab
matplotlib.use('Agg')

print(f'matplotlib version: {matplotlib.__version__}')
print(f'pandas version: {pd.__version__}')

matplotlib version: 3.10.0
pandas version: 2.2.2


In [ ]:
# Define paths

RESULTS_DIR = Path('/content/drive/MyDrive/HRC_Research/results/occlusion_benchmark')
FIGURES_DIR = Path('/content/drive/MyDrive/HRC_Research/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

OCCLUSION_CSV    = RESULTS_DIR / 'occlusion_results.csv'
CONSENSUS_CSV    = RESULTS_DIR / 'consensus_benchmark_results.csv'

# Verify files exist before proceeding
for f in [OCCLUSION_CSV, CONSENSUS_CSV]:
    assert f.exists(), f'FILE NOT FOUND: {f}'
    print(f'Found: {f}')
print('All paths verified.')

Found: /content/drive/MyDrive/HRC_Research/results/occlusion_benchmark/occlusion_results.csv
Found: /content/drive/MyDrive/HRC_Research/results/occlusion_benchmark/consensus_benchmark_results.csv
All paths verified.


In [ ]:
# Load locked results from CSV

occ_df = pd.read_csv(OCCLUSION_CSV)
con_df = pd.read_csv(CONSENSUS_CSV)

# X axis values (occlusion rates as integers for plotting)
x_rates = [0, 10, 20, 30, 50]

# Read mean values directly from the CSV columns
stgcn_means  = occ_df['stgcn_mean'].values
ctrgcn_means = occ_df['ctrgcn_mean'].values
con_means    = con_df['con_mean'].values

print('ST-GCN means  :', [f'{v:.2f}' for v in stgcn_means])
print('CTR-GCN means :', [f'{v:.2f}' for v in ctrgcn_means])
print('Consensus means:', [f'{v:.2f}' for v in con_means])

# LOCKED improvement values — hardcoded, must NOT be recomputed from CSV
locked_improvements = {
    '0%':  1.70,
    '10%': 4.48,
    '20%': 5.81,
    '30%': 1.16,
    '50%': 0.68,
}
imp_values = list(locked_improvements.values())
print('Locked improvements:', imp_values)

ST-GCN means  : ['61.22', '49.05', '40.48', '39.01', '23.47']
CTR-GCN means : ['69.05', '52.76', '37.04', '26.94', '16.22']
Consensus means: ['70.75', '57.24', '46.29', '40.17', '24.15']
Locked improvements: [1.7, 4.48, 5.81, 1.16, 0.68]


In [ ]:
# IEEE publication style settings

IEEE_RC = {
    'font.family':        'serif',
    'font.serif':         ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':          10,
    'axes.labelsize':     10,
    'axes.titlesize':     10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    8,
    'figure.dpi':         300,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.02,
    'lines.linewidth':    1.2,
    'lines.markersize':   4,
}

plt.rcParams.update(IEEE_RC)

# IEEE single-column width = 3.5 inches
IEEE_W = 3.5
IEEE_H = 2.625  # 4:3 ratio — standard for single-column figures

print('IEEE style applied.')
print(f'Figure size: {IEEE_W} x {IEEE_H} inches at 300 DPI')

IEEE style applied.
Figure size: 3.5 x 2.625 inches at 300 DPI


In [ ]:
# Figure 1: Accuracy Degradation Curves

fig1, ax1 = plt.subplots(figsize=(IEEE_W, IEEE_H))

# Plot three lines
ax1.plot(x_rates, stgcn_means,
         color='#1f77b4', linestyle='--', marker='o',
         label='ST-GCN')

ax1.plot(x_rates, ctrgcn_means,
         color='#ff7f0e', linestyle='--', marker='s',
         label='CTR-GCN')

ax1.plot(x_rates, con_means,
         color='#2ca02c', linestyle='-', linewidth=1.8, marker='^',
         label='Consensus (ours)')

# Crossover vertical line
ax1.axvline(x=20, color='grey', linestyle=':', linewidth=0.9)
ax1.text(20.8, 4, 'Crossover\npoint',
         fontsize=7, color='grey', va='bottom')

# Axes
ax1.set_xlabel('Occlusion Rate (%)')
ax1.set_ylabel('Top-1 Accuracy (%)')
ax1.set_xlim(-2, 54)
ax1.set_ylim(0, 80)
ax1.set_xticks(x_rates)
ax1.set_xticklabels([f'{r}%' for r in x_rates])
ax1.yaxis.set_major_locator(ticker.MultipleLocator(10))

# Grid
ax1.grid(True, linewidth=0.4, alpha=0.6)

# Legend
ax1.legend(loc='upper right', framealpha=0.9, edgecolor='none')

# Remove top and right spines (IEEE style)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

plt.tight_layout()

# Save
out1 = FIGURES_DIR / 'accuracy_degradation_curves.png'
fig1.savefig(out1)
plt.show()
print(f'Saved: {out1}')

Saved: /content/drive/MyDrive/HRC_Research/figures/accuracy_degradation_curves.png


In [ ]:
# Figure 2: Consensus Improvement Bars

fig2, ax2 = plt.subplots(figsize=(IEEE_W, IEEE_H))

x_labels = ['0%', '10%', '20%', '30%', '50%']
x_pos    = np.arange(len(x_labels))

bars = ax2.bar(x_pos, imp_values,
               color='#2ca02c', width=0.55,
               edgecolor='white', linewidth=0.5)

# Value labels on top of each bar
for bar, val in zip(bars, imp_values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.08,
        f'+{val:.2f} pp',
        ha='center', va='bottom', fontsize=7.5
    )

# Axes
ax2.set_xlabel('Occlusion Rate (%)')
ax2.set_ylabel('Improvement over Best Baseline (pp)')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_labels)
ax2.set_ylim(0, 7)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(1))

# Y-axis grid only
ax2.grid(True, axis='y', linewidth=0.4, alpha=0.6)
ax2.set_axisbelow(True)

# Remove top and right spines (IEEE style)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()

# Save
out2 = FIGURES_DIR / 'consensus_improvement_bars.png'
fig2.savefig(out2)
plt.show()
print(f'Saved: {out2}')

Saved: /content/drive/MyDrive/HRC_Research/figures/consensus_improvement_bars.png


In [ ]:
# Final confirmation

import os

figures = [
    FIGURES_DIR / 'accuracy_degradation_curves.png',
    FIGURES_DIR / 'consensus_improvement_bars.png',
]

print('Output verification:')
for f in figures:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f.name}: {size_kb:.1f} KB')
print('Done. Both figures saved to Drive.')

Output verification:
  accuracy_degradation_curves.png: 102.8 KB
  consensus_improvement_bars.png: 59.5 KB
Done. Both figures saved to Drive.
